In [1]:
from first import Tensor, label_locals, topo, trace
from tests import tests, rnd, tt
import torch


In [2]:
# tests()

In [3]:
a = Tensor([1, 2, 3, 4, 5, 6], (2, 3))
print(a.tolist())

[[1, 2, 3], [4, 5, 6]]


In [4]:
# x = Tensor([[1.0, 2.0], [3.0, 4.0]])
# w = Tensor([[10.0, 20.0], [30.0, 40.0]])
# b = Tensor([100.0, 200.0])

# y = x @ w + b
# z = y * y

# label_locals(locals())  # names x, w, b, y, z from the variables holding them
# print("\n".join(trace(z)))


In [5]:
a = Tensor([[1.0, 2.0], [3.0, 4.0]])
b = Tensor([[10.0, 20.0], [30.0, 40.0]])
z = (a + b) * b
z.backward()

ta = torch.tensor(a.tolist(), dtype=torch.float64, requires_grad=True)
tb = torch.tensor(b.tolist(), dtype=torch.float64, requires_grad=True)
tz = (ta + tb) * tb
tz.backward(torch.ones_like(tz))  # same seed as ours: all 1.0

assert z.tolist() == tz.tolist()
assert a.grad == [10, 20, 30, 40] == ta.grad.flatten().tolist()  # [10, 20, 30, 40]
assert b.grad == [21, 42, 63, 84] == tb.grad.flatten().tolist()  # [21, 42, 63, 84]

# the reuse case: x feeds mul twice, so d(x*x)/dx = 2x
x = Tensor([2.0, 3.0])
(x * x).backward()
print(x.grad)
assert x.grad == [4.0, 6.0]
print("ok")


[4.0, 6.0]
ok


In [6]:
x = torch.tensor([2.0, 3.0], requires_grad=True)
(x * x).backward(torch.ones((2,)))
x.grad

tensor([4., 6.])

In [7]:
def tg(t):
    return torch.tensor(t.tolist(), dtype=torch.float64, requires_grad=True)


for s1, s2 in [
    ((2, 3), (3,)),
    ((2, 3), (2, 1)),
    ((2, 3, 4), (3, 1)),
    ((2, 1, 4), (3, 4)),
    ((5,), ()),
    ((1,), (7, 8)),
    ((2, 3), (2, 3)),
]:
    for op in [lambda p, q: p + q, lambda p, q: p * q, lambda p, q: p - q]:
        a, b = rnd(s1), rnd(s2)
        o = op(a, b)
        o.backward()

        ta, tb = tg(a), tg(b)
        to = op(ta, tb)
        to.backward(torch.ones_like(to))

        assert torch.allclose(
            torch.tensor(a.grad, dtype=torch.float64), ta.grad.flatten(), atol=1e-12
        ), (s1, s2)
        assert torch.allclose(
            torch.tensor(b.grad, dtype=torch.float64), tb.grad.flatten(), atol=1e-12
        ), (s1, s2)
print("ok")


ok


In [8]:
x = Tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])  # (2,3)
b = Tensor([10.0, 20.0, 30.0])  # (3,)
(x + b).backward()

print(x.grad)  # [1, 1, 1, 1, 1, 1]   — six slots, one each
print(b.grad)  # [2, 2, 2]            — each value was used twice


[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
[2.0, 2.0, 2.0]
